# Phase 2A — Focal Loss Cross-Validation

**What changed from Phase 1:** One line — `nn.CrossEntropyLoss` is replaced with `FocalLoss(gamma=2)`.
Everything else is identical: same backbone, same 5-fold patient-level splits, same LP setup, same evaluation.

**Why:** Phase 1 showed R3A sensitivity = 0.517 and threshold tuning couldn't move it — the model's raw R3A probabilities are too low. Focal loss redirects training attention away from easy, already-correct examples (mostly R0) toward the hard, uncertain examples (R2, R3A). After training, the model should assign higher raw probabilities to R3A cases, which then lets threshold tuning work.

**What we compare at the end:** Phase 1 OOF (CE + class weights) vs Phase 2A OOF (focal + class weights).

In [ ]:
import os, sys, json, copy, math, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import PIL.ImageFile
PIL.ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, cohen_kappa_score, confusion_matrix

os.chdir(os.path.dirname(os.path.abspath('phase2a_focal_loss.ipynb')))
sys.path.insert(0, '.')
print('Working directory:', os.getcwd())

# ── Config (identical to Phase 1) ─────────────────────────────────────────────
N_FOLDS       = 5
MAX_EPOCHS    = 50
PATIENCE      = 10
BATCH_SIZE    = 64
INPUT_SIZE    = 224
LR            = 1.25e-3
MIN_LR        = 1e-6
WARMUP_EPOCHS = 5
WEIGHT_DECAY  = 0.05
NUM_CLASSES   = 4
CLASS_WEIGHTS = [1.0, 1.7851, 9.5294, 15.6774]
SEED          = 42
CLASSES       = ['R0', 'R1', 'R2', 'R3A']

# ── Focal loss config ─────────────────────────────────────────────────────────
FOCAL_GAMMA   = 2.0   # standard value from Lin et al. 2017

HF_REPO    = 'YukunZhou/RETFound_dinov2_meh'
HF_FILE    = 'RETFound_dinov2_meh.pth'
CV_OUTPUT  = Path('output_dir/phase2a_cv')
CV_OUTPUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Config: {N_FOLDS}-fold CV, focal_gamma={FOCAL_GAMMA}, patience={PATIENCE}')

## Focal loss — the only change from Phase 1

Standard cross-entropy: `loss = -log(p_true)`  
Focal loss: `loss = -(1 - p_true)^γ × log(p_true)`

The `(1 - p_true)^γ` factor shrinks the loss for easy (high-confidence) examples and keeps it nearly unchanged for hard (low-confidence) examples. With γ=2 and class weights applied on top:

| p_true | (1-p)² | Relative to CE |
|--------|--------|----------------|
| 0.95   | 0.003  | −99.7% of CE   |
| 0.70   | 0.09   | −91% of CE     |
| 0.30   | 0.49   | −51% of CE     |
| 0.10   | 0.81   | −19% of CE     |

R0 images are mostly at 0.90+ confidence → their gradient is almost zero.  
R3A images are at 0.10–0.20 confidence → their gradient is nearly undiminished.

In [ ]:
class FocalLoss(nn.Module):
    """
    Multiclass focal loss with optional per-class weights.

    Normalisation: divides by sum of class weights in the batch (same convention
    as PyTorch's CrossEntropyLoss with weight=), so the loss magnitude is
    comparable to Phase 1 CE and the same LR works without retuning.
    """
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        # log-softmax → log p for every class
        log_p  = F.log_softmax(logits, dim=1)              # (B, C)
        # gather log p of the TRUE class
        log_pt = log_p.gather(1, targets.view(-1, 1)).squeeze(1)  # (B,)
        # p_t = probability of true class
        pt     = log_pt.exp()                               # (B,)
        # focusing factor: shrinks loss for easy (high-p) examples
        focal_weight = (1.0 - pt) ** self.gamma             # (B,)

        if self.weight is not None:
            alpha = self.weight[targets]                    # (B,)
            focal_weight = focal_weight * alpha
            # Normalise by sum of per-sample class weights (matches PyTorch CE)
            loss = -(focal_weight * log_pt).sum() / alpha.sum()
        else:
            loss = -(focal_weight * log_pt).mean()

        return loss


# ── Sanity checks ──────────────────────────────────────────────────────────────
torch.manual_seed(0)
_w       = torch.tensor(CLASS_WEIGHTS)
_logits  = torch.randn(16, 4)
_targets = torch.randint(0, 4, (16,))

# 1. gamma=0 focal loss should match PyTorch weighted CE exactly
_fl0 = FocalLoss(gamma=0.0, weight=_w)
_ce  = nn.CrossEntropyLoss(weight=_w)
_diff = abs(float(_fl0(_logits, _targets)) - float(_ce(_logits, _targets)))
print(f'gamma=0 vs weighted CE diff: {_diff:.2e}  (must be < 1e-5): {"PASS" if _diff < 1e-5 else "FAIL"}')

# 2. gamma=2 loss should be strictly smaller than gamma=0 (focusing reduces loss)
_fl2 = FocalLoss(gamma=2.0, weight=_w)
_less = float(_fl2(_logits, _targets)) < float(_fl0(_logits, _targets))
print(f'gamma=2 loss < gamma=0 loss: {"PASS" if _less else "FAIL"}')

# 3. Show focusing table
print()
print('Focusing factor (1-p)^2 at different confidence levels:')
for p in [0.95, 0.80, 0.60, 0.40, 0.20, 0.10]:
    factor = (1 - p) ** FOCAL_GAMMA
    print(f'  p_true={p:.2f}  factor={factor:.4f}  (loss scaled to {factor*100:.1f}% of CE)')

In [ ]:
# ── Load splits (identical to Phase 1) ────────────────────────────────────────
GRADE = {'R0': 0, 'R1': 1, 'R2': 2, 'R3A': 3}

df_all = pd.read_csv('labels/splits.csv')
df_all['grade_int'] = df_all['retinopathy'].map(GRADE)

df_cv   = df_all[df_all['split'].isin(['train', 'val'])].copy()
df_test = df_all[df_all['split'] == 'test'].copy()

pat_grade = df_cv.groupby('code')['grade_int'].max().reset_index()
pat_grade.columns = ['code', 'strat_grade']
patient_ids   = pat_grade['code'].values
patient_strat = pat_grade['strat_grade'].values

# Fold assignments — same SEED → same splits as Phase 1
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_assignments = {}
for fold_idx, (_, val_pat_idx) in enumerate(skf.split(patient_ids, patient_strat)):
    for pid in patient_ids[val_pat_idx]:
        fold_assignments[pid] = fold_idx
pat_grade['fold'] = pat_grade['code'].map(fold_assignments)

df_cv = df_cv.reset_index(drop=True)
df_cv['cv_idx'] = df_cv.index

print(f'CV pool : {len(df_cv)} images | {len(patient_ids)} patients')
print(f'Test set: {len(df_test)} images | {df_test["code"].nunique()} patients')
print('Same folds as Phase 1 (SEED=42) — direct comparison is valid.')

In [ ]:
import argparse
from util.datasets import build_transform

_aug_args = argparse.Namespace(
    input_size=INPUT_SIZE, color_jitter=None,
    aa='rand-m9-mstd0.5-inc1', reprob=0.25, remode='pixel', recount=1,
)
train_transform = build_transform('train', _aug_args)
eval_transform  = build_transform('val',   _aug_args)

class RetinopathyDataset(Dataset):
    def __init__(self, records, transform):
        self.records   = records
        self.transform = transform
    def __len__(self):  return len(self.records)
    def __getitem__(self, idx):
        path, label = self.records[idx]
        return self.transform(Image.open(path).convert('RGB')), label

def make_records(df_subset):
    return [(row.image_path, row.grade_int) for row in df_subset.itertuples()]

print('Dataset helpers ready.')

In [ ]:
import timm
from huggingface_hub import hf_hub_download
from timm.layers import trunc_normal_

def load_backbone(device, num_classes=NUM_CLASSES, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)
    model = timm.create_model(
        'vit_large_patch14_dinov2.lvd142m',
        pretrained=True, img_size=INPUT_SIZE,
        num_classes=num_classes, drop_path_rate=0.2,
    )
    ckpt_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE)
    ckpt      = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    ckpt_model = ckpt['teacher']
    ckpt_model = {k.replace('backbone.', ''): v for k, v in ckpt_model.items()}
    ckpt_model = {k.replace('mlp.w12.', 'mlp.fc1.'): v for k, v in ckpt_model.items()}
    ckpt_model = {k.replace('mlp.w3.',  'mlp.fc2.'): v for k, v in ckpt_model.items()}
    state = model.state_dict()
    drop  = [k for k in ckpt_model if k in state and ckpt_model[k].shape != state[k].shape]
    for k in drop:
        print(f'  Skipping mismatched: {k}')
        del ckpt_model[k]
    model.load_state_dict(ckpt_model, strict=False)
    trunc_normal_(model.head.weight, std=2e-5)
    nn.init.zeros_(model.head.bias)
    for name, param in model.named_parameters():
        param.requires_grad = ('head' in name)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)')
    return model.to(device)

print('Verifying backbone load...')
_m = load_backbone(DEVICE)
print('OK.')
del _m
torch.cuda.empty_cache()

In [ ]:
# ── Training helpers (identical to Phase 1 except criterion is now FocalLoss) ─

class EarlyStopping:
    def __init__(self, patience, model):
        self.patience   = patience
        self.best_auroc = -float('inf')
        self.counter    = 0
        self.best_head  = copy.deepcopy(model.head.state_dict())
    def step(self, auroc, model):
        if auroc != auroc: auroc = 0.0
        if auroc > self.best_auroc:
            self.best_auroc = auroc
            self.counter    = 0
            self.best_head  = copy.deepcopy(model.head.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience
    def restore(self, model):
        model.head.load_state_dict(self.best_head)


def get_lr(epoch, warmup_epochs, max_epochs, lr, min_lr):
    if epoch < warmup_epochs:
        return lr * (epoch + 1) / warmup_epochs
    t = (epoch - warmup_epochs) / max(1, max_epochs - warmup_epochs)
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * t))


def train_epoch(model, loader, optimizer, criterion, device, scaler, epoch,
                warmup_epochs, max_epochs, lr, min_lr):
    model.train()
    cur_lr = get_lr(epoch, warmup_epochs, max_epochs, lr, min_lr)
    for pg in optimizer.param_groups:
        pg['lr'] = cur_lr
    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss = criterion(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(labels)
    return total_loss / len(loader.dataset), cur_lr


@torch.no_grad()
def eval_fold(model, loader, device):
    model.eval()
    all_labels, all_probs = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
        probs = torch.softmax(logits, dim=1).cpu().float()
        all_labels.append(labels)
        all_probs.append(probs)
    return torch.cat(all_labels).numpy(), torch.cat(all_probs).numpy()


def compute_metrics(labels, probs, preds=None):
    """Compute metrics. preds defaults to argmax(probs) if not supplied."""
    if preds is None:
        preds = probs.argmax(axis=1)
    probs_f64 = probs.astype(np.float64)
    probs_f64 = probs_f64 / probs_f64.sum(axis=1, keepdims=True)
    try:
        auroc = roc_auc_score(labels, probs_f64, multi_class='ovr', average='macro')
    except Exception:
        auroc = float('nan')
    acc   = accuracy_score(labels, preds)
    kappa = cohen_kappa_score(labels, preds, weights='quadratic')
    cm    = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    sens, spec = [], []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]; fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fn - fp
        sens.append(tp / (tp + fn) if (tp + fn) > 0 else float('nan'))
        spec.append(tn / (tn + fp) if (tn + fp) > 0 else float('nan'))
    return {'auroc': auroc, 'accuracy': acc, 'kappa': kappa,
            'macro_sensitivity': np.nanmean(sens), 'macro_specificity': np.nanmean(spec),
            'sensitivity': np.array(sens), 'specificity': np.array(spec)}


print('Helpers defined.')

## 5-fold CV training loop with focal loss

The only difference from Phase 1 is the criterion:
```python
# Phase 1:  criterion = nn.CrossEntropyLoss(weight=weight_tensor)
# Phase 2A: criterion = FocalLoss(gamma=2.0, weight=weight_tensor)
```

Everything else — splits, backbone, head initialisation, optimizer, LR schedule, early stopping — is identical. This ensures any metric difference is attributable solely to the loss function.

In [ ]:
weight_tensor = torch.tensor(CLASS_WEIGHTS, dtype=torch.float).to(DEVICE)
criterion_cv  = FocalLoss(gamma=FOCAL_GAMMA, weight=weight_tensor)

oof_labels_all = np.zeros(len(df_cv), dtype=np.int64)
oof_probs_all  = np.zeros((len(df_cv), NUM_CLASSES), dtype=np.float32)
fold_results   = []

for fold in range(N_FOLDS):
    print(f'\n{"="*60}')
    print(f'  FOLD {fold+1}/{N_FOLDS}  [focal loss γ={FOCAL_GAMMA}]')
    print(f'{"="*60}')

    val_pats   = pat_grade[pat_grade['fold'] == fold]['code'].values
    train_pats = pat_grade[pat_grade['fold'] != fold]['code'].values
    df_fold_train = df_cv[df_cv['code'].isin(train_pats)]
    df_fold_val   = df_cv[df_cv['code'].isin(val_pats)]
    val_cv_indices = df_fold_val['cv_idx'].values
    print(f'  Train: {len(df_fold_train)} images | Val: {len(df_fold_val)} images')

    ds_train = RetinopathyDataset(make_records(df_fold_train), train_transform)
    ds_val   = RetinopathyDataset(make_records(df_fold_val),   eval_transform)
    ds_test  = RetinopathyDataset(make_records(df_test),       eval_transform)

    loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=False)
    loader_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
    loader_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

    model     = load_backbone(device=DEVICE, seed=SEED + fold)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=WEIGHT_DECAY,
    )
    scaler  = torch.cuda.amp.GradScaler()
    stopper = EarlyStopping(patience=PATIENCE, model=model)

    t_start = time.time()
    for epoch in range(MAX_EPOCHS):
        tr_loss, cur_lr = train_epoch(
            model, loader_train, optimizer, criterion_cv, DEVICE, scaler,
            epoch, WARMUP_EPOCHS, MAX_EPOCHS, LR, MIN_LR,
        )
        val_labels, val_probs = eval_fold(model, loader_val, DEVICE)
        m = compute_metrics(val_labels, val_probs)
        elapsed = time.time() - t_start
        print(f'  ep {epoch:02d} | lr={cur_lr:.2e} | loss={tr_loss:.4f} | '
              f'AUROC={m["auroc"]:.4f} | sens={m["macro_sensitivity"]:.4f} | '
              f'elapsed={elapsed:.0f}s')
        if stopper.step(m['auroc'], model):
            print(f'  Early stop at epoch {epoch} (best AUROC={stopper.best_auroc:.4f})')
            break

    stopper.restore(model)
    print(f'  Best val AUROC: {stopper.best_auroc:.4f}')

    oof_labels, oof_probs = eval_fold(model, loader_val, DEVICE)
    oof_labels_all[val_cv_indices] = oof_labels
    oof_probs_all[val_cv_indices]  = oof_probs

    test_labels_fold, test_probs_fold = eval_fold(model, loader_test, DEVICE)

    np.save(CV_OUTPUT / f'fold_{fold}_oof_probs.npy',   oof_probs)
    np.save(CV_OUTPUT / f'fold_{fold}_oof_labels.npy',  oof_labels)
    np.save(CV_OUTPUT / f'fold_{fold}_test_probs.npy',  test_probs_fold)
    np.save(CV_OUTPUT / f'fold_{fold}_test_labels.npy', test_labels_fold)

    m_fold = compute_metrics(oof_labels, oof_probs)
    fold_results.append({
        'fold': fold, 'best_auroc': stopper.best_auroc,
        'oof_auroc': m_fold['auroc'], 'oof_kappa': m_fold['kappa'],
        'oof_macro_sens': m_fold['macro_sensitivity'],
        'oof_macro_spec': m_fold['macro_specificity'],
    })
    print(f'  OOF AUROC: {m_fold["auroc"]:.4f}  Kappa: {m_fold["kappa"]:.4f}')

    del model
    torch.cuda.empty_cache()

print('\n' + '='*60)
print('All folds complete.')
np.save(CV_OUTPUT / 'oof_labels_all.npy', oof_labels_all)
np.save(CV_OUTPUT / 'oof_probs_all.npy',  oof_probs_all)
with open(CV_OUTPUT / 'fold_results.json', 'w') as f:
    json.dump(fold_results, f, indent=2)
print(f'Saved to {CV_OUTPUT}/')

## Results: Phase 1 (CE) vs Phase 2A (Focal Loss)

The cell below loads both sets of OOF predictions and builds a side-by-side comparison.
It also applies the Youden thresholds found in Phase 2C to both models, so we can see
whether focal loss improves the *raw* predictions and whether threshold tuning on top adds further gain.

**What to look for:**
- Does R3A OOF sensitivity increase? If yes, focal loss is working.
- Does the AUROC hold? (Focal loss should not hurt ranking, only sharpen the decision boundary.)
- Does the Youden threshold applied to Phase 2A further improve R3A? (Phase 2C thresholds were tuned on Phase 1 OOF — re-tuning on Phase 2A OOF may give additional gain.)

In [ ]:
from sklearn.metrics import roc_curve

P1_OUTPUT = Path('output_dir/phase1_cv')

# ── Load predictions ──────────────────────────────────────────────────────────
p1_oof_labels = np.load(P1_OUTPUT / 'oof_labels_all.npy')
p1_oof_probs  = np.load(P1_OUTPUT / 'oof_probs_all.npy').astype(np.float64)
p1_oof_probs  = p1_oof_probs / p1_oof_probs.sum(axis=1, keepdims=True)

p2_oof_labels = np.load(CV_OUTPUT / 'oof_labels_all.npy')
p2_oof_probs  = np.load(CV_OUTPUT / 'oof_probs_all.npy').astype(np.float64)
p2_oof_probs  = p2_oof_probs / p2_oof_probs.sum(axis=1, keepdims=True)

# Test ensemble
p2_test_probs_list = [
    np.load(CV_OUTPUT / f'fold_{f}_test_probs.npy').astype(np.float64)
    for f in range(N_FOLDS)
]
p2_test_labels = np.load(CV_OUTPUT / 'fold_0_test_labels.npy')
p2_test_probs  = np.mean(p2_test_probs_list, axis=0)
p2_test_probs  = p2_test_probs / p2_test_probs.sum(axis=1, keepdims=True)

np.save(CV_OUTPUT / 'test_ensemble_probs.npy',  p2_test_probs)
np.save(CV_OUTPUT / 'test_ensemble_labels.npy', p2_test_labels)

# ── Youden thresholds on Phase 2A OOF ────────────────────────────────────────
youden_thr_p2 = []
for i in range(NUM_CLASSES):
    fpr, tpr, thrs = roc_curve((p2_oof_labels == i).astype(int), p2_oof_probs[:, i])
    j = tpr - fpr
    youden_thr_p2.append(float(thrs[j.argmax()]))

try:
    with open('output_dir/phase2c_thresholds/thresholds.json') as f:
        p2c = json.load(f)
    youden_thr_p1 = [p2c['youden_thresholds'][c] for c in CLASSES]
except Exception:
    youden_thr_p1 = [0.4907, 0.3160, 0.1252, 0.1463]

print(f'Youden thresholds — Phase 1 OOF : {[f"{t:.4f}" for t in youden_thr_p1]}')
print(f'Youden thresholds — Phase 2A OOF: {[f"{t:.4f}" for t in youden_thr_p2]}')

def apply_thresholds(probs, thresholds):
    thresholds = np.array(thresholds)
    ratios = probs / thresholds
    above  = probs > thresholds
    return np.where(above.any(axis=1), ratios.argmax(axis=1), probs.argmax(axis=1))

# ── Four variants: (model) × (decision rule) ─────────────────────────────────
m_p1_argmax  = compute_metrics(p1_oof_labels, p1_oof_probs)
m_p1_youden  = compute_metrics(p1_oof_labels, p1_oof_probs,
                                apply_thresholds(p1_oof_probs, youden_thr_p1))
m_p2_argmax  = compute_metrics(p2_oof_labels, p2_oof_probs)
m_p2_youden  = compute_metrics(p2_oof_labels, p2_oof_probs,
                                apply_thresholds(p2_oof_probs, youden_thr_p2))

p1_test_probs  = np.load(P1_OUTPUT / 'test_ensemble_probs.npy')
p1_test_probs  = p1_test_probs / p1_test_probs.sum(axis=1, keepdims=True)
p1_test_labels = np.load(P1_OUTPUT / 'test_ensemble_labels.npy')

mt_p1_argmax = compute_metrics(p1_test_labels, p1_test_probs)
mt_p1_youden = compute_metrics(p1_test_labels, p1_test_probs,
                                apply_thresholds(p1_test_probs, youden_thr_p1))
mt_p2_argmax = compute_metrics(p2_test_labels, p2_test_probs)
mt_p2_youden = compute_metrics(p2_test_labels, p2_test_probs,
                                apply_thresholds(p2_test_probs, youden_thr_p2))

# cols: (name, oof_metrics, test_metrics)
cols = [
    ('Phase1+Argmax',  m_p1_argmax,  mt_p1_argmax),
    ('Phase1+Youden',  m_p1_youden,  mt_p1_youden),
    ('Phase2A+Argmax', m_p2_argmax,  mt_p2_argmax),
    ('Phase2A+Youden', m_p2_youden,  mt_p2_youden),
]

# ── Print comparison table ────────────────────────────────────────────────────
for split_name, get_m in [('OOF (990 patients)',          lambda c: c[1]),
                           ('TEST ENSEMBLE (702 images)', lambda c: c[2])]:
    print()
    print('=' * 90)
    print(f'  {split_name}')
    print('=' * 90)
    print(f'{"Metric":<22}' + ''.join(f'{c[0]:>20}' for c in cols))
    print('-' * 90)
    for key, label in [('auroc','AUROC'), ('kappa','Kappa'),
                       ('macro_sensitivity','Macro sens'),
                       ('macro_specificity','Macro spec')]:
        print(f'{label:<22}' + ''.join(f'{get_m(c)[key]:>20.4f}' for c in cols))
    print()
    for i, cls in enumerate(CLASSES):
        print(f'{cls+" sens":<22}' + ''.join(f'{get_m(c)["sensitivity"][i]:>20.4f}' for c in cols))
        print(f'{cls+" spec":<22}' + ''.join(f'{get_m(c)["specificity"][i]:>20.4f}' for c in cols))

# ── Save summary ──────────────────────────────────────────────────────────────
summary = {
    'focal_gamma': FOCAL_GAMMA,
    'youden_thresholds_p2a': {c: youden_thr_p2[i] for i, c in enumerate(CLASSES)},
    'oof':  {name: {'auroc': m_oof['auroc'],
                    'macro_sensitivity': m_oof['macro_sensitivity'],
                    'sensitivity': m_oof['sensitivity'].tolist()}
             for name, m_oof, _ in cols},
    'test': {name: {'auroc': m_test['auroc'],
                    'macro_sensitivity': m_test['macro_sensitivity'],
                    'sensitivity': m_test['sensitivity'].tolist()}
             for name, _, m_test in cols},
}
with open(CV_OUTPUT / 'phase2a_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print(f'\nSummary saved to {CV_OUTPUT}/phase2a_summary.json')